<a href="https://colab.research.google.com/github/dineshaiacademy/5-day-ai-bootcamp/blob/main/Day%201%20-%20LLM%20Fundamentals/Learning/llm_fundamentals_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 LLM Fundamentals — Hands-On with Gemini

Welcome to Day 1! This notebook builds a working understanding of how Large Language Models (LLMs) actually behave — not just theory, but real, runnable calls against Google's Gemini API.

**By the end of this notebook, you will be able to:**
1. Securely load an API key — locally in VS Code, or in Google Colab
2. Make your first call to an LLM and read the response
3. Control model behavior with `temperature`, `top_p`, and `max_output_tokens`
4. Steer the model's behavior with a system instruction
5. Hold a multi-turn conversation using chat history
6. Stream a response token-by-token
7. Understand tokens and the context window

Follow the cells **in order, top to bottom** — each step builds directly on the one before it.

## ✅ Prerequisites

- A free Google account
- A Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey)
- Python 3.9+ (already available in Colab; use your bootcamp `venv` locally)

> No prior LLM experience needed — we start from the basics.

## 🔑 Step 1 — Get Your Gemini API Key

1. Go to **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)** and sign in with your Google account.
2. Click **Create API key** and copy it.
3. Store it as **`GAISTUDIO_API_KEY`**:
   - **VS Code / local**: open the `.env` file at the repo root and set `GAISTUDIO_API_KEY=your-key-here`
   - **Google Colab**: click the 🔑 key icon in the left sidebar → **Add new secret** → name it `GAISTUDIO_API_KEY` → paste your key → enable notebook access

Never paste your key directly into a code cell — the helper below reads it securely from whichever environment you're in.

In [ ]:
!pip install -q google-genai python-dotenv

## ⚙️ Step 2 — Load the Key and Connect to Gemini

The `get_secret()` function below works the same whether you're in Colab or VS Code — it checks Colab's secret store first, then falls back to your local `.env` file.

In [ ]:
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")

if not GAISTUDIO_API_KEY:
    raise ValueError(
        "GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file."
    )

print("✅ API key loaded successfully.")

In [ ]:
from google import genai

client = genai.Client(api_key=GAISTUDIO_API_KEY)
MODEL = "gemini-2.5-flash"

print(f"✅ Connected. Using model: {MODEL}")

## 💬 Step 3 — Your First LLM Call

At its core, an LLM takes text in (a **prompt**) and returns text out (a **completion**) — predicted one token at a time based on patterns learned from training data.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Explain what a Large Language Model is, in exactly two sentences.",
)

print(response.text)

## 🎛️ Step 4 — Controlling Behavior with Parameters

Three parameters shape how the model generates text:

| Parameter | What it controls | Typical range |
|---|---|---|
| `temperature` | Randomness — low = focused/deterministic, high = creative/varied | 0.0 – 2.0 |
| `top_p` | Limits sampling to the smallest set of tokens whose combined probability exceeds `top_p` | 0.0 – 1.0 |
| `max_output_tokens` | Hard cap on response length | model-dependent |

The cell below runs the same prompt at `temperature=0.0` and then `temperature=1.5` so you can compare the outputs directly.

In [ ]:
from google.genai import types

for temp in [0.0, 1.5]:
    response = client.models.generate_content(
        model=MODEL,
        contents="Give me one creative name for a coffee shop.",
        config=types.GenerateContentConfig(
            temperature=temp,
            max_output_tokens=30,
        ),
    )
    print(f"temperature={temp} -> {response.text.strip()}")

## 🧭 Step 5 — System Instructions

A **system instruction** sets the model's persona and ground rules before it sees the user's prompt — it's how you steer tone, role, and constraints consistently across every call.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="How do I center a div?",
    config=types.GenerateContentConfig(
        system_instruction="You are a terse senior engineer. Answer in at most 2 lines, no fluff.",
    ),
)

print(response.text)

## 🔄 Step 6 — Multi-Turn Conversations

LLMs are **stateless** — each call has no memory of previous ones. A chat "remembers" only because the full conversation history is re-sent with every request. Gemini's `chats` API handles this for you.

In [ ]:
chat = client.chats.create(model=MODEL)

reply_1 = chat.send_message("My name is Dinesh and I'm building an AI bootcamp.")
print("Model:", reply_1.text)

reply_2 = chat.send_message("What did I say I'm building?")
print("Model:", reply_2.text)

## ⚡ Step 7 — Streaming Responses

Instead of waiting for the entire response, you can stream it token-by-token — this is what powers the "typing" effect in apps like ChatGPT and Gemini.

In [ ]:
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="Count from 1 to 5, one number per line.",
):
    print(chunk.text, end="", flush=True)

## 🔢 Step 8 — Tokens and the Context Window

LLMs don't read words — they read **tokens** (roughly 4 characters of English text each). Every model has a maximum **context window**: the total tokens it can hold across your prompt + its response in a single call.

In [ ]:
prompt = "The quick brown fox jumps over the lazy dog."
token_count = client.models.count_tokens(model=MODEL, contents=prompt)

print(f"Prompt: {prompt!r}")
print(f"Token count: {token_count.total_tokens}")

## 🎯 Recap

You've now covered the core mechanics behind every LLM-powered app:

| Concept | What you learned |
|---|---|
| Secure key handling | `.env` locally, Colab Secrets in the cloud |
| Basic call | prompt in → completion out |
| Sampling controls | `temperature`, `top_p`, `max_output_tokens` |
| System instructions | steering tone and role |
| Multi-turn chat | history re-sent on every call |
| Streaming | token-by-token delivery |
| Tokens & context window | the model's real unit of text, and its hard limit |

**Next:** head to `Day 1 - LLM Fundamentals/Projects/` to apply these concepts in a real Streamlit app, or copy a starter from `Templates/`.